In [1]:
import json
import os
from pathlib import Path

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from qdrant_client import AsyncQdrantClient

from enterprise_rag.router import route_query
from enterprise_rag.retrieval import retrieve_and_response
from enterprise_rag.generation import rag_formatted_response
from enterprise_rag.pipeline import handle_query
from enterprise_rag.evaluation import evaluate_router

load_dotenv()
nest_asyncio.apply()

qdrant_url = os.getenv("QDRANT_URL")
if qdrant_url:
    qdrant = AsyncQdrantClient(url=qdrant_url, api_key=os.getenv("QDRANT_API_KEY"))
else:
    # no QDRANT_URL configured -> run Qdrant in-process, no server or API key needed
    qdrant = AsyncQdrantClient(location=":memory:")

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

/Users/bahloulia/Downloads/agentic_software/Entreprise Grade RAG/.venv/lib/python3.11/site-packages/qdrant_client/async_qdrant_remote.py:231: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


## Data Adhesion

In [ ]:
chunks = [
    {
        "content": "Full-time employees accrue 15 days of paid time off (PTO) per year during their first three years of employment, increasing to 20 days after three years of continuous service.",
        "source": "employee_handbook.pdf",
        "page": 12,
    },
    {
        "content": "Employees are eligible for up to 12 weeks of paid parental leave following the birth, adoption, or foster placement of a child, provided they have completed at least 90 days of employment.",
        "source": "employee_handbook.pdf",
        "page": 18,
    },
    {
        "content": "Open enrollment for medical, dental, and vision benefits occurs once a year in November. Employees may also enroll within 30 days of a qualifying life event such as marriage or the birth of a child.",
        "source": "benefits_guide.pdf",
        "page": 4,
    },
    {
        "content": "Uber Technologies reported total revenue of $37.3 billion for fiscal year 2023, representing a 17% increase compared to fiscal year 2022.",
        "source": "uber_10k_2023.pdf",
        "page": 52,
    },
    {
        "content": "Lyft's operating expenses for fiscal year 2023 totaled $4.6 billion, driven primarily by insurance costs and sales and marketing spend.",
        "source": "lyft_10k_2023.pdf",
        "page": 47,
    },
    {
        "content": "Uber's 10-K filing identifies driver classification litigation as a material risk factor, noting that reclassification of drivers as employees in certain jurisdictions could materially increase operating costs.",
        "source": "uber_10k_2023.pdf",
        "page": 21,
    },
]
for chunk in chunks:
    result=route_query(chunk['content'])
    print(result['action'])

    ingest_documents(
        qdrant,
        result,
        chunks: list[dict],
        tokenizer,
        model,
        vector_size: int,
    )

HUMAN_RESOURCES_QUERY
HUMAN_RESOURCES_QUERY
HUMAN_RESOURCES_QUERY
10K_DOCUMENT_QUERY
10K_DOCUMENT_QUERY
10K_DOCUMENT_QUERY


'\ningest_documents(\n    qdrant: AsyncQdrantClient,\n    collection_name: str,\n    chunks: list[dict],\n    tokenizer,\n    model,\n    vector_size: int,\n)'

## Dynamic Routing

`route_query()` asks Claude to classify a user question into one of three
categories -- `ANTHROPIC_QUERY`, `10K_DOCUMENT_QUERY`, or `WEB_SEARCH` --
before any retrieval happens. `handle_query()` uses that decision to send
the query down the right path: a Qdrant vector search (with the retrieved
chunks' source and page carried through for citations) followed by a
Claude-generated, citation-backed answer.

![Dynamic routing pipeline](../assets/router_architecture_simple.svg)

| Component | Lives in |
|---|---|
| `route_query` | `enterprise_rag/router.py` |
| `retrieve_and_response` | `enterprise_rag/retrieval.py` |
| `rag_formatted_response` | `enterprise_rag/generation.py` |
| `handle_query` | `enterprise_rag/pipeline.py` |

In [2]:
user_query = "What was Uber's 2023 revenue?"
answer = await handle_query(qdrant, user_query)
answer

Route: 10K_DOCUMENT_QUERY
Reason: This question asks for specific company financial data from Uber's annual reports, which would be found in 10-K filings.


ResponseHandlingException: [Errno 8] nodename nor servname provided, or not known

## Evaluating the Router

`route_query()` is an LLM classifier, so we shouldn't just trust it -- we
should measure it. Below we run it against 100 hand-labeled queries
(`notebooks/data/router_eval_queries.json`, ~34/33/33 split across the
three route categories) and score the predictions with `evaluate_router()`
(`enterprise_rag/evaluation.py`): overall accuracy, per-class
precision/recall/F1, a confusion matrix, and the individual queries it
got wrong.


In [ ]:
eval_dataset = json.loads(Path("data/router_eval_queries.json").read_text())
results = evaluate_router(eval_dataset)

accuracy = results["report"]["accuracy"]
print(f"Accuracy: {accuracy:.1%} ({len(eval_dataset)} queries)")

Accuracy: 100.0% (100 queries)


In [ ]:
pd.DataFrame(results["report"]).T

,precision,recall,f1-score,support
10K_DOCUMENT_QUERY,1.0,1.0,1.0,33.0
ANTHROPIC_QUERY,1.0,1.0,1.0,34.0
WEB_SEARCH,1.0,1.0,1.0,33.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,100.0
weighted avg,1.0,1.0,1.0,100.0


In [ ]:
pd.DataFrame(
    results["confusion_matrix"],
    index=[f"true: {label}" for label in results["labels"]],
    columns=[f"pred: {label}" for label in results["labels"]],
)

,pred: 10K_DOCUMENT_QUERY,pred: ANTHROPIC_QUERY,pred: WEB_SEARCH
true: 10K_DOCUMENT_QUERY,33,0,0
true: ANTHROPIC_QUERY,0,34,0
true: WEB_SEARCH,0,0,33


In [ ]:
errors_df = pd.DataFrame(
    [p for p in results["predictions"] if p["predicted"] != p["expected"]]
)
errors_df

""
